# Phase 4: Targeted Mask-and-Repair with Verification-Style Regeneration

**Contributor:** Pragnya Suresh

**What this does:**
1. Takes hallucinated sentences flagged in Phase 3
2. Masks the lowest-confidence entity spans with `[BLANK_n]` placeholders
3. Prompts Llama-3.1-8B-Instruct through a Chain-of-Thought verification loop to fill the blanks
4. Evaluates repair quality using NLI contradiction scores against Wikipedia ground truth
5. Saves all results for Phase 5 handoff

**Runtime:** Use **A100 GPU**. If unavailable, use L4 or T4 (see memory note at bottom). Expected ~30-60 min total.

## 0. Setup

In [ ]:
!git clone https://github.com/pragnya-suresh18/LLM-Hallucination-Detection.git
%cd LLM-Hallucination-Detection

In [ ]:
!pip install -q transformers accelerate datasets selfcheckgpt spacy scikit-learn matplotlib pandas numpy scipy bert-score sentencepiece protobuf
!python -m spacy download en_core_web_sm -q

In [ ]:
from huggingface_hub import login
login()  # paste your HF token when prompted (needs Llama 3.1 access)

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    total_mem = getattr(props, "total_memory", getattr(props, "total_mem", 0))
    print(f"Memory: {total_mem / 1e9:.1f} GB")

In [ ]:
# Verify Phase 3 outputs exist
from pathlib import Path
required = ["data/phase3_hybrid_flags.json", "data/sentences_with_splits.csv"]
for f in required:
    exists = Path(f).exists()
    print(f"{'OK' if exists else 'MISSING'}: {f}")
    if not exists:
        raise FileNotFoundError(f"{f} not found — run earlier phases first")

## 1. Load Data & Collect Flagged Sentences

In [ ]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

DATA_DIR = Path("data")
FIG_DIR = DATA_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
MAX_NEW_TOKENS = 192
LOGPROB_THRESHOLD = -3.0

print("Loading flags...")
with open(DATA_DIR / "phase3_hybrid_flags.json") as f:
    flags_data = json.load(f)

print("Loading sentences...")
df = pd.read_csv(DATA_DIR / "sentences_with_splits.csv")

# Lookup: (passage_id, sentence_idx) -> wiki_bio_text
wiki_lookup = {}
for _, row in df.iterrows():
    key = (int(row["passage_id"]), int(row["sentence_idx"]))
    wiki_lookup[key] = row.get("wiki_bio_text", "")

print(f"Loaded {len(df)} sentences, {df['passage_id'].nunique()} passages")

In [ ]:
repair_queue = []

for passage in flags_data:
    pid = passage["passage_id"]
    passage_sentences = passage["sentences"]
    passage_text = " ".join(s["sentence"] for s in passage_sentences)

    for sent in passage_sentences:
        if not sent["is_hallucinated"]:
            continue

        risky_spans = [
            span for span in sent.get("flagged_spans", [])
            if span["mean_logprob"] < LOGPROB_THRESHOLD
        ]

        repair_queue.append({
            "passage_id": pid,
            "sentence_idx": sent["sentence_idx"],
            "sentence": sent["sentence"],
            "split": sent["split"],
            "label": sent["label"],
            "hybrid_score": sent["hybrid_score"],
            "passage_context": passage_text,
            "risky_spans": risky_spans,
            "repair_mode": "span_mask" if risky_spans else "full_sentence",
        })

total_flagged = sum(1 for p in flags_data for s in p["sentences"] if s["is_hallucinated"])
span_mask_count = sum(1 for r in repair_queue if r["repair_mode"] == "span_mask")
full_sent_count = sum(1 for r in repair_queue if r["repair_mode"] == "full_sentence")

print(f"{total_flagged} flagged sentences, {len(repair_queue)} in repair queue")
print(f"  span_mask mode : {span_mask_count}")
print(f"  full_sentence  : {full_sent_count}")

## 2. Load Llama 3.1 8B Instruct

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

print(f"Loading tokenizer from {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

use_fa2 = False
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    try:
        import flash_attn  # noqa: F401
        use_fa2 = True
    except ImportError:
        pass
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8) else torch.float16
attn_impl = "flash_attention_2" if use_fa2 else "sdpa"
print(f"Loading model — dtype={dtype}, attention={attn_impl} ...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    attn_implementation=attn_impl,
    low_cpu_mem_usage=True,
    device_map="auto",
)
model.eval()

if hasattr(torch, "compile"):
    print("Applying torch.compile (first call will be slow, subsequent calls faster) ...")
    model = torch.compile(model, mode="reduce-overhead")

n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model ready — {n_params:.1f}B parameters on {device}")

if torch.cuda.is_available():
    mem_used = torch.cuda.memory_allocated() / 1e9
    props = torch.cuda.get_device_properties(0)
    mem_total = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / 1e9
    print(f"GPU memory: {mem_used:.1f} / {mem_total:.1f} GB")

## 3. Prompt Templates & Generation

In [ ]:
def build_masked_sentence(sentence, risky_spans):
    """Replace risky entity spans with [BLANK_1], [BLANK_2], etc."""
    masked = sentence
    sorted_spans = sorted(risky_spans, key=lambda s: s["start_char"], reverse=True)
    blank_map = {}
    for i, span in enumerate(sorted(risky_spans, key=lambda s: s["start_char"])):
        blank_map[span["text"]] = f"[BLANK_{i+1}]"

    for span in sorted_spans:
        start = span["start_char"]
        end = span["end_char"]
        blank_label = blank_map[span["text"]]
        masked = masked[:start] + blank_label + masked[end:]

    return masked, blank_map


def build_verification_prompt(sentence, masked_sentence, blank_map, passage_context):
    """Compact Chain-of-Thought verification prompt."""
    blanks_desc = "; ".join(f'{blank}="{text}"' for text, blank in blank_map.items())
    prompt = f"""Fix factual errors in a biography sentence. Blanks mark low-confidence spans.

Context: {passage_context}

Masked: {masked_sentence}
Blanks: {blanks_desc}

For each blank, verify the correct value, then output the full corrected sentence on the last line prefixed with "CORRECTED: "."""
    return prompt


def build_full_sentence_repair_prompt(sentence, passage_context):
    """Compact repair prompt for sentences without specific spans."""
    prompt = f"""Fix factual errors in this biography sentence.

Context: {passage_context}

Sentence: {sentence}

Identify incorrect claims, determine correct facts, then output the corrected sentence on the last line prefixed with "CORRECTED: "."""
    return prompt


@torch.no_grad()
def generate_response(prompt):
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=2048).to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
    )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def extract_corrected_sentence(response):
    """Extract the line starting with 'CORRECTED: ' from the model response."""
    for line in reversed(response.split("\n")):
        line = line.strip()
        if line.upper().startswith("CORRECTED:"):
            return line[len("CORRECTED:"):].strip().strip('"').strip("'")
    lines = [l.strip() for l in response.split("\n") if l.strip()]
    return lines[-1] if lines else ""

## 4. Run Repair Pipeline

In [ ]:
# Benchmark
print("Benchmarking on 2 sentences ...")
t0 = time.time()
for item in repair_queue[:2]:
    if item["repair_mode"] == "span_mask" and item["risky_spans"]:
        masked, bmap = build_masked_sentence(item["sentence"], item["risky_spans"])
        prompt = build_verification_prompt(item["sentence"], masked, bmap, item["passage_context"][:800])
    else:
        prompt = build_full_sentence_repair_prompt(item["sentence"], item["passage_context"][:800])
    generate_response(prompt)
elapsed = time.time() - t0
per_sent = elapsed / 2
est_total = per_sent * len(repair_queue) / 60
print(f"~{per_sent:.1f}s per sentence → estimated total: {est_total:.1f} min")

In [ ]:
print(f"Starting repair for {len(repair_queue)} sentences ...")
results = []
t_start = time.time()

for item in tqdm(repair_queue, desc="Repairing"):
    sentence = item["sentence"]
    context = item["passage_context"][:800]

    if item["repair_mode"] == "span_mask" and item["risky_spans"]:
        masked_sentence, blank_map = build_masked_sentence(sentence, item["risky_spans"])
        prompt = build_verification_prompt(sentence, masked_sentence, blank_map, context)
    else:
        masked_sentence = None
        blank_map = {}
        prompt = build_full_sentence_repair_prompt(sentence, context)

    response = generate_response(prompt)
    corrected = extract_corrected_sentence(response)

    results.append({
        "passage_id": item["passage_id"],
        "sentence_idx": item["sentence_idx"],
        "split": item["split"],
        "label": item["label"],
        "hybrid_score": item["hybrid_score"],
        "repair_mode": item["repair_mode"],
        "original_sentence": sentence,
        "masked_sentence": masked_sentence,
        "blank_map": {v: k for k, v in blank_map.items()} if blank_map else {},
        "corrected_sentence": corrected,
        "full_response": response,
        "num_spans_masked": len(item["risky_spans"]),
    })

elapsed_total = (time.time() - t_start) / 60
print(f"\nRepair complete in {elapsed_total:.1f} min")
print(f"Total repaired: {len(results)} sentences")

In [ ]:
# Save repaired sentences
OUTPUT_PATH = DATA_DIR / "phase4_repaired.json"
with open(OUTPUT_PATH, "w") as f:
    json.dump(results, f, indent=2)
size_mb = OUTPUT_PATH.stat().st_size / 1e6
print(f"Saved → {OUTPUT_PATH} ({size_mb:.1f} MB)")

## 5. Unload Llama & Load NLI Model for Evaluation

In [ ]:
# Free GPU memory from Llama before loading DeBERTa
del model
del tokenizer
torch.cuda.empty_cache()

if torch.cuda.is_available():
    print(f"GPU memory after unload: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
from transformers import DebertaV2ForSequenceClassification, DebertaV2Tokenizer

NLI_MODEL = "potsawee/DeBERTa-v3-large-mnli"
NLI_BATCH_SIZE = 32

print(f"Loading NLI model: {NLI_MODEL} ...")
nli_tokenizer = DebertaV2Tokenizer.from_pretrained(NLI_MODEL)
nli_model = DebertaV2ForSequenceClassification.from_pretrained(NLI_MODEL)
nli_model.eval()
nli_model.to(device)
print("NLI model loaded.")

@torch.no_grad()
def nli_contradiction_scores_batch(sentence_pairs):
    """Return P(contradiction) for a batch of (sentence, reference) pairs."""
    sents = [p[0] for p in sentence_pairs]
    refs = [p[1] for p in sentence_pairs]
    inputs = nli_tokenizer(
        sents, refs,
        return_tensors="pt", truncation=True, max_length=512, padding=True,
    ).to(device)
    logits = nli_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)
    return probs[:, 1].cpu().tolist()

## 6. Evaluate: Before vs After Repair

In [ ]:
test_results = [r for r in results if r["split"] == "test"]
print(f"Evaluating {len(test_results)} repaired test sentences ...")

valid_results = []
for r in test_results:
    key = (r["passage_id"], r["sentence_idx"])
    wiki_text = wiki_lookup.get(key, "")
    if wiki_text and r["corrected_sentence"]:
        valid_results.append((r, wiki_text))

print(f"  {len(valid_results)} have valid wiki references")

orig_scores = []
rep_scores = []
for i in tqdm(range(0, len(valid_results), NLI_BATCH_SIZE), desc="NLI eval (batched)"):
    batch = valid_results[i : i + NLI_BATCH_SIZE]
    orig_pairs = [(r["original_sentence"], wiki) for r, wiki in batch]
    rep_pairs = [(r["corrected_sentence"], wiki) for r, wiki in batch]
    orig_scores.extend(nli_contradiction_scores_batch(orig_pairs))
    rep_scores.extend(nli_contradiction_scores_batch(rep_pairs))

eval_records = []
for idx, (r, wiki_text) in enumerate(valid_results):
    eval_records.append({
        "passage_id": r["passage_id"],
        "sentence_idx": r["sentence_idx"],
        "label": r["label"],
        "repair_mode": r["repair_mode"],
        "original_sentence": r["original_sentence"],
        "corrected_sentence": r["corrected_sentence"],
        "original_contradiction": orig_scores[idx],
        "repaired_contradiction": rep_scores[idx],
        "improvement": orig_scores[idx] - rep_scores[idx],
    })

eval_df = pd.DataFrame(eval_records)
print(f"Evaluated {len(eval_df)} sentences")

In [ ]:
if len(eval_df) > 0:
    mean_orig = eval_df["original_contradiction"].mean()
    mean_rep = eval_df["repaired_contradiction"].mean()
    mean_improvement = eval_df["improvement"].mean()
    improved_count = int((eval_df["improvement"] > 0).sum())
    worsened_count = int((eval_df["improvement"] < 0).sum())
    unchanged_count = int((eval_df["improvement"] == 0).sum())

    hallu_df = eval_df[eval_df["label"] == 1]
    hallu_mean_orig = hallu_df["original_contradiction"].mean() if len(hallu_df) > 0 else 0
    hallu_mean_rep = hallu_df["repaired_contradiction"].mean() if len(hallu_df) > 0 else 0
    hallu_improved = int((hallu_df["improvement"] > 0).sum()) if len(hallu_df) > 0 else 0

    print("\n" + "=" * 60)
    print("MITIGATION RESULTS (Test Set)")
    print("=" * 60)
    print(f"Sentences evaluated      : {len(eval_df)}")
    print(f"Mean contradiction (orig): {mean_orig:.4f}")
    print(f"Mean contradiction (rep) : {mean_rep:.4f}")
    print(f"Mean improvement         : {mean_improvement:.4f}")
    print(f"Improved / Worsened / Same: {improved_count} / {worsened_count} / {unchanged_count}")
    print(f"\nHallucinated sentences only ({len(hallu_df)}):")
    print(f"  Mean contradiction (orig): {hallu_mean_orig:.4f}")
    print(f"  Mean contradiction (rep) : {hallu_mean_rep:.4f}")
    print(f"  Improved                 : {hallu_improved}/{len(hallu_df)}")
else:
    print("WARNING: No sentences evaluated")

## 7. Figures

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", font_scale=1.1)

if len(eval_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(eval_df["original_contradiction"], bins=30, alpha=0.6,
                 color="#F44336", label="Original", edgecolor="black", linewidth=0.3)
    axes[0].hist(eval_df["repaired_contradiction"], bins=30, alpha=0.6,
                 color="#4CAF50", label="Repaired", edgecolor="black", linewidth=0.3)
    axes[0].set_title("NLI Contradiction Score Distribution (Test)")
    axes[0].set_xlabel("P(contradiction) with Wikipedia")
    axes[0].set_ylabel("Count")
    axes[0].legend()

    axes[1].hist(eval_df["improvement"], bins=30, color="#5C6BC0",
                 edgecolor="black", linewidth=0.3, alpha=0.8)
    axes[1].axvline(0, color="red", linestyle="--", linewidth=1.2)
    axes[1].axvline(mean_improvement, color="black", linestyle="--", linewidth=1.2,
                    label=f"Mean={mean_improvement:.3f}")
    axes[1].set_title("Improvement per Sentence (Higher = Better)")
    axes[1].set_xlabel("Δ P(contradiction): Original − Repaired")
    axes[1].set_ylabel("Count")
    axes[1].legend()

    plt.suptitle("Phase 4: Mitigation — Before vs After Repair", fontweight="bold")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "phase4_mitigation_eval.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
if len(eval_df) > 0 and eval_df["repair_mode"].nunique() > 1:
    fig, ax = plt.subplots(figsize=(8, 5))
    modes = eval_df["repair_mode"].unique()
    mode_improvements = [eval_df[eval_df["repair_mode"] == m]["improvement"].mean() for m in modes]
    mode_counts = [len(eval_df[eval_df["repair_mode"] == m]) for m in modes]
    ax.bar(
        [f"{m}\n(n={c})" for m, c in zip(modes, mode_counts)],
        mode_improvements,
        color=["#5C6BC0", "#EF5350"],
        edgecolor="black", linewidth=0.5,
    )
    ax.axhline(0, color="gray", linestyle="--")
    ax.set_ylabel("Mean Improvement (Δ contradiction)")
    ax.set_title("Improvement by Repair Mode")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "phase4_by_repair_mode.png", dpi=150, bbox_inches="tight")
    plt.show()

## 8. Qualitative Examples

In [ ]:
if len(eval_df) > 0:
    sorted_df = eval_df.sort_values("improvement", ascending=False)

    print("TOP 5 MOST IMPROVED:")
    print("=" * 80)
    for _, row in sorted_df.head(5).iterrows():
        print(f"[Δ={row['improvement']:+.3f}] mode={row['repair_mode']}")
        print(f"  ORIG : {row['original_sentence']}")
        print(f"  FIXED: {row['corrected_sentence']}")
        print()

    print("\nTOP 5 MOST WORSENED:")
    print("=" * 80)
    for _, row in sorted_df.tail(5).iterrows():
        print(f"[Δ={row['improvement']:+.3f}] mode={row['repair_mode']}")
        print(f"  ORIG : {row['original_sentence']}")
        print(f"  FIXED: {row['corrected_sentence']}")
        print()

## 9. Save All Results for Phase 5 Handoff

In [ ]:
if len(eval_df) > 0:
    eval_output = {
        "model": MODEL_NAME,
        "logprob_threshold": LOGPROB_THRESHOLD,
        "test_sentences_evaluated": len(eval_df),
        "mean_contradiction_original": float(mean_orig),
        "mean_contradiction_repaired": float(mean_rep),
        "mean_improvement": float(mean_improvement),
        "improved_count": improved_count,
        "worsened_count": worsened_count,
        "unchanged_count": unchanged_count,
        "hallucinated_only": {
            "count": int(len(hallu_df)),
            "mean_contradiction_original": float(hallu_mean_orig),
            "mean_contradiction_repaired": float(hallu_mean_rep),
            "improved_count": hallu_improved,
        },
        "per_sentence": eval_records,
    }
else:
    eval_output = {"error": "No sentences could be evaluated"}

EVAL_PATH = DATA_DIR / "phase4_eval_results.json"
with open(EVAL_PATH, "w") as f:
    json.dump(eval_output, f, indent=2)
print(f"Saved eval results → {EVAL_PATH}")

In [ ]:
# Verify all output files exist
print("\nPhase 4 output files:")
for f in ["phase4_repaired.json", "phase4_eval_results.json"]:
    path = DATA_DIR / f
    if path.exists():
        size = path.stat().st_size / 1e6
        print(f"  OK  {f} ({size:.1f} MB)")
    else:
        print(f"  MISSING  {f}")

for f in ["phase4_mitigation_eval.png", "phase4_by_repair_mode.png"]:
    path = FIG_DIR / f
    print(f"  {'OK' if path.exists() else 'MISSING'}  figures/{f}")

## 10. Push Results to Repo

In [ ]:
# Uncomment and run to push results back to your repo
# !git add data/phase4_repaired.json data/phase4_eval_results.json data/figures/phase4_*.png
# !git commit -m "Phase 4: mitigation repair results and evaluation"
# !git push

## Phase 5 Handoff Notes

**Files for your teammates:**

| File | What it contains |
|------|------------------|
| `data/phase4_repaired.json` | Every repaired sentence with `original_sentence`, `corrected_sentence`, `label`, `repair_mode`, `full_response` |
| `data/phase4_eval_results.json` | Aggregate metrics + per-sentence NLI contradiction scores (original vs repaired) |
| `data/phase3_hybrid_flags.json` | ALL sentences (including untouched accurate ones) — needed for collateral damage analysis |
| `data/sentences_with_splits.csv` | Has `wiki_bio_text` (Wikipedia ground truth) for each passage |

**What Phase 5 needs to compute:**
1. Hallucination rate before vs after repair (absolute + relative reduction)
2. Sentence-level factual accuracy before vs after
3. **Collateral damage**: check if any originally accurate sentences (label=0) that were falsely flagged and repaired became worse
4. Aggregate: total hallucinations removed, total new hallucinations introduced, net improvement